# Moduli space limits in JAXVacua

**What's in this notebook?** This notebook is a theory reference for the moduli space limits implemented in JAXVacua: the Large Complex Structure (LCS) limit, the coni-LCS limit near a conifold singularity, and the coni-LCS bulk approximation.
For each limit, the notebook explains the underlying prepotential, shows how to initialise the corresponding JAXVacua model, and reproduces a literature benchmark.

**Contents:**
- [LCS limits](#LCS-limits) — polynomial + instanton prepotential; $h^{1,2}=2$ example reproducing [2501.03984](https://arxiv.org/abs/2501.03984); Newton convergence plot
- [Coni-LCS limits](#coni-LCS-limits) — logarithmic prepotential near a conifold; basis change via `jvc.get_basis_change`; $h^{1,1}=99$, $h^{1,2}=3$ example from [2009.03312](https://arxiv.org/abs/2009.03312)
- [Bulk approximation at coniLCS](#Bulk-approximation-at-coniLCS) — effective bulk prepotential integrating out the conifold modulus

(*Created:* Andreas Schachner, June 25, 2024)

**Related notebooks:**
- [3_cytools_interface.ipynb](../01_basics/3_cytools_interface.ipynb) — loading models from CYTools and performing basis changes
- [13_coniLCS.ipynb](13_coniLCS.ipynb) — coni-LCS models: analytical PFV solutions, `compute_zcf`, linearised expansion
- [16_freezer_and_flux_utils.ipynb](16_freezer_and_flux_utils.ipynb) — freezing moduli with the `Freezer` class; integrating out directions in the scalar potential

**Prerequisites:** Basic familiarity with JAXVacua (see [2_jaxvacua_overview.ipynb](../01_basics/2_jaxvacua_overview.ipynb)) and CYTools (see [3_cytools_interface.ipynb](../01_basics/3_cytools_interface.ipynb)).

## Imports

In [ ]:
# General imports
import warnings
import numpy as np
from tqdm.auto import tqdm
from scipy.optimize import root

# JAX imports
from jax import jit, vmap
import jax
import jax.numpy as jnp
jax.config.update("jax_enable_x64", True)

# Plotting
import seaborn as sn
import matplotlib.pyplot as plt
cmap = sn.color_palette("viridis", as_cmap=True)

# JAXVacua
import jaxvacua as jvc

warnings.filterwarnings('ignore')

## LCS limits

### Introduction

At Large Complex Structure (LCS), the prepotential can be expressed as

$$
    F_{\mathrm{LCS}}(X) = F_{\mathrm{poly}}(X) + F_{\mathrm{inst}}(X)\, .
$$

Here, the polynomial part $F_{\mathrm{poly}}$ of the LCS prepotential $F_{\mathrm{LCS}}$
can be expressed in terms of the periods $X^I=(X^0,X^i)$ as

$$
    F_{\mathrm{poly}}(X) = -\frac{1}{6X^0}\widetilde{\kappa}_{ijk}X^iX^jX^k+\frac{1}{2}a_{ij}X^iX^j
        +b_{i}X^i\, X^0 + \dfrac{\text{i}}{2}\tilde{\xi}\, (X^0)^2\, ,
$$

Here, $\widetilde{\kappa}_{ijk}$ are the triple intersection numbers of
the mirror dual Calabi-Yau threefold $\widetilde{X}$.
Here, we defined

$$
    a_{ij} = \dfrac{1}{2}\begin{cases}
                            \widetilde{\kappa}_{iij} & i\geq j\\[0.3em]
                            \widetilde{\kappa}_{ijj} & i<j
                        \end{cases} \, , \quad 
    b_i = \dfrac{1}{24} \int_{\tilde{D}^i}\, c_2(\widetilde{X})\, , \quad  
    \tilde{\xi}=\frac{\zeta(3)\, \chi(\widetilde{X})}{(2\pi)^3}\, .
$$

The instanton part $F_{\mathrm{inst}}$ of the LCS prepotential $F_{\mathrm{LCS}}$
can be expressed in terms of the periods $X^I=(X^0,X^i)$ as

$$
    F_{\mathrm{inst}}(X) = -\frac{(X^0)^2}{(2\pi\mathrm{i})^3}\, \sum_{q\in\mathcal{M}(\widetilde{X})}\, 
    n_q^{0}\, \text{Li}_3\left (\text{e}^{2\pi \text{i} q_i X^i / X^0}\right )\; , \quad 
    \text{Li}_3\left (x\right )=\sum_{m=1}^{\infty}\, \dfrac{x^{m}}{m^{3}}\, .
$$

Here the sum is performed over all effective curve classes $q\in\mathcal{M}(\widetilde{X})$
in the Mori cone $\mathcal{M}(\widetilde{X})$ of the mirror dual manifold $\widetilde{X}$.
Here, the $n_q^{0}$ are the genus-0 Gopakumar-Vafa (GV) invariants which can be computed
systematically using methods described in [hep-th/9308122](https://arxiv.org/pdf/hep-th/9308122.pdf).

The infinite sum appearing in the poly-logarithm $\text{Li}_3$ can be rewritten to arrive at

$$
    \sum_{q\in\mathcal{M}(\widetilde{X})}\, n_q^{0}\, \text{Li}_3\left (\text{e}^{2\pi \text{i} q_i X^i / X^0}\right )
    = \sum_{q\in\mathcal{M}(\widetilde{X})}\, N_q\, \text{e}^{2\pi \text{i} q_i X^i / X^0}
$$

in terms of genus-0 Gromov-Witten (GW) invariants $N_q$. We typically work with the latter to simplify the calculation.


The above functions are implemented in the class `periods_LCS` which is automatically called in `periods` class if we specify `limit="LCS"`.

### Example at $h^{1,2} = 2$ - $\mathbb{CP}[1,1,1,6,9]$

We load a pre-built LCS model using stored topological data. The key constructor parameters are:
- `h12`: number of complex structure moduli $h^{1,2}$
- `model_ID`: selects among the pre-stored models of the given type and `h12` value (here the default `model_type="KS"` refers to models in the Kreuzer–Skarke database)
- `limit="LCS"`: activates the LCS prepotential class (`periods_LCS`), which evaluates $F = F_{\rm poly} + F_{\rm inst}$
- `maximum_degree`: truncates the GV instanton sum at curves up to this total degree; `maximum_degree=1` keeps only the two leading worldsheet corrections, which is sufficient for a quick benchmark but shifts the minimum compared to the fully converged result

In [ ]:
model = jvc.FluxVacuaFinder(h12=2,model_ID=1,maximum_degree=1, limit="LCS")
model

This is the $\mathbb{CP}^4_{[1,1,1,6,9]}$ hypersurface discussed in [1912.10047](https://arxiv.org/pdf/1912.10047). The topological data is accessible through the `lcs_tree` attribute. We can verify that the intersection numbers, the $a$-matrix $a_{ij}$, and the $b$-vector $b_i$ match the expected values:

In [ ]:
model.lcs_tree.a_matrix = jnp.array([[4.5,1.5],[1.5,0.]])

In [ ]:
model.lcs_tree.intnums,model.lcs_tree.a_matrix,model.lcs_tree.b_vector

JAXVacua uses the genus-0 Gromov–Witten (GW) invariants $N_q$ to evaluate $F_{\rm inst}$ (obtained from the GV invariants $n_q^0$ via $N_q = \sum_{d|q} n_{q/d}^0 / d^3$). With `maximum_degree=1`, only the two simplest curve classes on the mirror contribute:
- $q = (1,0)$: GV invariant $n_q^0 = 540$
- $q = (0,1)$: GV invariant $n_q^0 = 3$

`gv_invariants` holds the values $n_q^0$ and `gv_charges` holds the corresponding curve class vectors $q_i$. Including higher-degree curves (e.g. `maximum_degree=2` adds $q = (1,1), (2,0), (0,2), \ldots$) shifts the vacuum by a sub-leading correction — typically affecting only the 3rd significant digit for `maximum_degree` $\geq 2$.

In [ ]:
model.lcs_tree.gv_invariants,model.lcs_tree.gv_charges

As a concrete example, we target the small-$|W_0|$ minimum from [2501.03984](https://arxiv.org/pdf/2501.03984) (Eq. 4.11). The initial moduli and flux values are taken from the paper. Note that [2501.03984] used a higher GV truncation degree; with our `maximum_degree=1` the Newton solver will converge to the nearby minimum of the truncated model, which differs slightly from the paper's exact vacuum.

In [ ]:
z0 = jnp.array([0.5+ 2.36817528j,  0.5+ 2.51175911j])
fluxes = jnp.array([4, 12,  2, -1,  0, -1, 36, -1,  0,  0,  1, -1.])
tau0 = 0.5+1j*1.48121567

At these starting values (taken from a higher-degree GV model), the $F$-term residuals are $O(1)$ — the initial guess is far from the `maximum_degree=1` vacuum:

In [ ]:
model.DW(z0,jnp.conj(z0),tau0,jnp.conj(tau0),fluxes)

We refine numerically using `newton_method_flux_vacua`. Key parameters:
- `step_size_Newton=1`: full Newton step; gives quadratic convergence near the minimum (reduce to ~0.1 for non-SUSY vacua where the basin of attraction may be small)
- `tol=1e-12`: terminates when $\max_I |D_I W| < 10^{-12}$
- `mode="SUSY"`: solves the system $D_I W = 0$ (supersymmetric vacuum); use `mode="non-SUSY"` to minimise $|\partial_i V|^2$ instead
- `solver_mode="complex"`: works in the complex moduli parameterisation (more efficient for SUSY vacua than the real-variable solver)

(**Note:** the first call triggers `jax.jit` compilation, which can take a few seconds.)

In [ ]:
moduli,tau,residual = model.newton_method_flux_vacua(z0,tau0,fluxes,step_size_Newton = 1,tol=1e-12,max_iters=100,mode="SUSY",solver_mode="complex")
print(f"|z-z0|: {np.abs(z0-moduli)}   |tau-tau0|: {np.abs(tau0-tau)}    |DW|: {residual}")

Newton's method has converged to the `maximum_degree=1` minimum. The moduli are shifted from the paper's starting values because the truncated GV sum changes the location of the minimum. The $F$-term conditions are now satisfied to machine precision:

In [ ]:
model.DW(moduli,jnp.conj(moduli),tau,jnp.conj(tau),fluxes)

### Newton convergence

`newton_method_flux_vacua` internally works in the **real representation**: `_convert_complex_to_real(z, z̄, τ, τ̄)` packs the real and imaginary parts of $(z, \tau)$ into a flat real array `x ∈ ℝ^{2(h^{1,2}+1)}`, and `DW_x(x, flux)` evaluates all $\{{\rm Re}(D_I W), {\rm Im}(D_I W)\}$ as a flat real vector. The Jacobian `dDW_x` is computed analytically via JAX. We can run the Newton step manually to visualise the (quadratic) convergence of $\max_I|D_I W|$:

In [ ]:
# Manual Newton iteration to track convergence
x = model._convert_complex_to_real(z0, jnp.conj(z0), tau0, jnp.conj(tau0))
n_iters = 12
residuals = []

for _ in range(n_iters):
    dw = model.DW_x(x, fluxes)
    residuals.append(float(jnp.max(jnp.abs(dw))))
    J = model.dDW_x(x, fluxes)
    x = x - jnp.linalg.solve(J, dw)

fig, ax = plt.subplots(figsize=(5.5, 3.5))
ax.semilogy(range(n_iters), residuals, "o-", color="steelblue", linewidth=2, markersize=5)
ax.set_xlabel("Newton iteration")
ax.set_ylabel(r"$\max_I\,|D_I W|$")
ax.set_title(r"Newton convergence — LCS example ($h^{1,2}=2$)")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

We extract the physical observables at the vacuum:
- `tadpole(fluxes)` returns the D3-brane tadpole contribution $N_{\rm flux} = \tfrac{1}{2}\int H_3 \wedge F_3$
- `superpotential_gauge_invariant` computes $W_0 = W/X^0$ in the gauge-invariant normalisation (divided by the period $X^0$, making it independent of the overall period normalisation); this is the quantity directly compared to literature values

In [ ]:
Nflux = model.tadpole(fluxes)
W = model.superpotential_gauge_invariant(moduli,tau,fluxes)
print(f"Nflux: {Nflux}    |W0|: {np.abs(W)}     ")

This matches the value $|W_0|\approx 5.5\times 10^{-5}$ sateted in Eq. (4.11) in [2501.03984](https://arxiv.org/pdf/2501.03984).
Note that small deviation are noticable here since we are only using the leading order GVs.
Sub-leading corrections shift the minimum and the value of $W_0$ slightly, but only affecting the 3rd significant digit.

## coni-LCS limits

### Introduction

To construct a Klebanov-Strassler throat region in a flux compactification, we must stabilize the complex structure moduli near a conifold locus.
To do so, we work near a boundary of moduli space where a conifold curve shrinks following the approach of [2009.03312](https://arxiv.org/abs/2009.03312), see also [2009.03325](https://arxiv.org/abs/2009.03325) for similar results. Here we will just sketch the construction, referring the reader to the original work for more details.

A **conifold singularity** is a locus in the complex structure moduli space of a Calabi-Yau threefold $X$ where a set of $n_{\mathrm{cf}}$ three-cycles, all of which lie in the same homology class $[\mathcal{C}] \in H_3(X,\mathbb{Z})$, shrink to zero volume. In an LCS patch, we can identify the complex structure moduli space of  $X$ with the  complexified  K\"ahler cone 
$\mathcal{K}(\widetilde{X})$
of the mirror threefold $\widetilde{X}$. In this picture, the conifold locus is identified with the facet of 
$\mathcal{K}(\widetilde{X})$ where a fixed set of curves $\mathcal{C}_{\mathrm{cf}}$ in some effective curve class  $\tilde{\mathbf{q}}^{\mathrm{cf}} \in \mathcal{M}(\widetilde{X})\cap H_2(\widetilde{X},\mathbb{Z})$, which we call the **conifold class**, shrink to zero size. We will defer discussion of how to actually find conifolds and how to compute $n_{\mathrm{cf}}$ to [2406.13751](https://arxiv.org/abs/2406.13751), and for now just assume the existence of some conifold class $\tilde{\mathbf{q}}^{\mathrm{cf}}$.
The volume of the curves $\mathcal{C}_{\mathrm{cf}}$ is measured by the absolute value of
$$
    z_{\text{cf}}= z^a \tilde{\mathbf{q}}^{\mathrm{cf}}_a\, ,
$$
which we will refer to as the **conifold modulus**. Denoting by ${\omega^a}_\alpha$ the generators of the lattice orthogonal to 
$\tilde{\mathbf{q}}^{\mathrm{cf}}$, it will be useful to parameterise moduli space according to
\begin{equation}
    z^a={\omega^a}_{\alpha} z^\alpha+ \xi^a z_{\text{cf}}\, ,
\end{equation}
with $a \in 1,\ldots,h^{2,1}$ and
$\alpha \in 2,\ldots,h^{2,1}$, in terms of the conifold modulus $z_{\mathrm{cf}}$ and the **bulk moduli** $z^\alpha$. Here, $\xi^a$ is an arbitrary constant vector  satisfying $\tilde{\mathbf{q}}^{\mathrm{cf}}_a\xi^a=1$.

In the following, it is convenient to work in a basis where $z_{\text{cf}}=z^1$ and $z_{\text{bulk}}^\alpha = z^\alpha$ for $\alpha=2,\ldots,h^{1,2}$.
The prepotential can then be written as an expansion in powers of $z_{\text{cf}}$ as (see [2009.03312](https://arxiv.org/abs/2009.03312))
\begin{equation}
    F_{\mathrm{coni-LCS}}(z) = n_{\text{cf}}\,\dfrac{z_{\text{cf}}^2}{4\pi\mathrm{i}}\,\ln(-2\pi\mathrm{i}z_{\text{cf}})
    +\sum_{n=0}^{\infty}\, \dfrac{F^{(n)}(z^\alpha)}{n!}\, z_{\text{cf}}^n
\end{equation}
where the higher order terms are obtained from
\begin{equation}
    F^{(n)}(z^\alpha) = (\partial_{z_{\text{cf}}}^n F_{\mathrm{poly}})\bigl |_{z_{\text{cf}}=0} - n_{\text{cf}}\dfrac{\hat{\zeta}(3-n)}{(2\pi\mathrm{i})^{3-n}}\,- \dfrac{1}{(2\pi\mathrm{i})^{3-n}}\, \sum_{[\mathcal{C}]\neq [\mathcal{C}_{\mathrm{cf}}]}\, n_{\mathcal{C}}^0\, (\beta_1^\mathcal{C})^n\, \mathrm{Li}_{3-n}(q^{\mathcal{C}})\bigl |_{z_{\text{cf}}=0}
\end{equation}
in terms of the polynomial prepotential
\begin{equation}
    F_{\mathrm{poly}}(z) = -\frac{1}{6}\widetilde{\kappa}_{abc}z^a z^b z^c+\frac{1}{2}a_{ab}z^a z^b
        +b_{a}z^a + \dfrac{\text{i}}{2}\tilde{\xi}\, .
\end{equation}

The prepotential $F_{\mathrm{coni-LCS}}(z)$ can be computed to any order in the expansion around $|z_{\text{cf}}|\ll 1$. For reasonably large $|z_{\text{cf}}| \gtrsim 10^{-5}$, it can be beneficial to check the full theory by taking the LCS prepotential without using Euler's reflection formula for the 3rd polylogarithm.


### Coni-LCS limts from CYTools

We study the original example of [2009.03312](https://arxiv.org/pdf/2009.03312) with $h^{1,1}=99$ and $h^{1,2}=3$.
The mirror dual CY can be defined as follows

In [ ]:
from cytools import Polytope
poly = Polytope([[-1,3,-2,-1],[1,-1,0,0],[-1,0,0,1],[-1,0,0,0],[-1,0,1,1],[-1,0,2,0],[-1,0,1,0]])
cy = poly.triangulate().get_cy()
cy

To find the conifold curve, we compute GV invariants to high degree and search for **nilpotent rays**: effective curve classes $\mathbf{q}$ such that $n_\mathbf{q}^0 \neq 0$ but $n_{m\mathbf{q}}^0 = 0$ for all multiples $m \geq 2$. A nilpotent ray signals a rational curve that does not have multi-cover contributions — physically, it shrinks to a point (a conifold singularity) as the corresponding Kähler modulus of the mirror $\widetilde{X}$ is taken to zero. The number of shrinking three-cycles is $n_{\rm cf} = n_\mathbf{q}^0 / 2$.

In [ ]:
gvs = cy.compute_gvs(max_deg=20)
grading_vector = gvs.grading_vec
gvs = gvs.dok
keys = gvs.keys()
curve_charges = np.array(list(keys))
GV_invariants = np.array(list(gvs.values()))

The nilpotent curve search: for each degree-1 curve (innermost Mori cone generator), check that all integer multiples $m\mathbf{q}$ for $m = 2, \ldots, 19$ have vanishing GV invariant. We record both the curve class and the GV value $n_\mathbf{q}^0$ (from which $n_{\rm cf} = n_\mathbf{q}^0 / 2$ follows):

In [ ]:
nilpotent_curves = []
for key in keys:
    q = np.array(key)
    degree = q@grading_vector
    if degree==1:
        flag=True
        for i in range(2,20):
            gv=gvs.get(tuple(i*q),0)
            if gv!=0:
                flag=False
                break
        if flag:    
            nilpotent_curves.append([q,gvs[key]])

nilpotent_curves

All three nilpotent curves coincide with the generators of the toric Mori cone (as expected for this model). The Mori cone generators are:

In [ ]:
mori = cy.toric_mori_cone(in_basis=True)
mori.extremal_rays()

In the example of [2009.03312](https://arxiv.org/pdf/2009.03312), the authors studied the conifold singularity arising when shrinking the curve associated with the class `[-1,1,0]`.
Let us define this as our conifold curve:

In [ ]:
conifold_curve = np.array([-1,1,0])

We can then find a suitable basis transformation in which this curve is represented by the class `[1,0,0]` by calling `jvc.get_basis_change`:

In [ ]:
basis_matrix = jvc.get_basis_change(conifold_curve)
basis_matrix

Let us check that it leads to the expected result

In [ ]:
conifold_curve@basis_matrix.T

The basis change matrix is not unique: any integer unimodular matrix $B$ satisfying $\tilde{\mathbf{q}}^{\rm cf} \cdot B^T = (1, 0, \ldots, 0)$ is valid. `get_basis_change` returns one canonical choice; another valid matrix is:

In [ ]:
basis_matrix = np.array([[0, 1, 1], [1, 1, 0], [0, 0, 1]])
conifold_curve@basis_matrix.T

### coni-LCS example in JAXVacua

We initialise the coni-LCS model. The additional parameters compared to the LCS case are:
- `use_cytools=True`, `mirror_cy=cy`: reads topological data directly from the CYTools `CalabiYau` object
- `Q=cy.h11()+cy.h12()+2`: total flux lattice dimension $Q = 2(h^{1,2}+1)$; by mirror symmetry $h^{1,1}_{\widetilde{X}} = h^{1,2}_X$, so $Q = 2(h^{1,1}+1) = 200$
- `ncf=2`: number of shrinking three-cycles $n_{\rm cf}$ at the conifold locus (read from $n_\mathbf{q}^0 = 4$, but the GV invariant counts the curve with multiplicity 2 in this example, giving $n_{\rm cf} = 2$)
- `basis_change=basis_matrix`: integer unimodular matrix rotating the moduli basis so the conifold class becomes $(1,0,0)$, as required by the coni-LCS prepotential ansatz

In [ ]:
basis_matrix = np.array([[0, 1, 1], [1, 1, 0], [0, 0, 1]])
conifold_curve = np.array([-1,1,0])

model = jvc.FluxVacuaFinder(h12=cy.h11(), 
                                    Q=cy.h11()+cy.h12()+2, 
                                    use_cytools=True, 
                                    mirror_cy=cy, 
                                    ncf=2,
                                    maximum_degree=6, 
                                    basis_change=basis_matrix, 
                                    limit="coniLCS",
                                    conifold_curve=conifold_curve,conifold_basis=True,
                                    prange=10,
                                    use_gvs=True)

We can check that the data obtained matches Eq. (4.42) in [2009.03312](https://arxiv.org/pdf/2009.03312):

In [ ]:
model.lcs_tree.a_matrix,model.lcs_tree.b_vector*24

We reproduce the solution from section 4.3 of [2009.03312](https://arxiv.org/pdf/2009.03312). In the **PFV (perfect flux vacuum) parametrisation**, the flux vector is expressed in terms of two integer vectors $M = (M_1, M_2, M_3)$ and $K = (K_1, K_2, K_3)$ of length $h^{1,2} = 3$. These encode the RR and NSNS flux quanta threading the conifold three-cycle and its dual, following the notation of [2009.03312]. Given $(M, K, \tau)$, the PFV approximation analytically solves $D_{z_{\rm cf}} W = 0$ for the conifold modulus $z_{\rm cf}$ and determines the bulk moduli $z^\alpha$ via `pfv_to_moduli`. We start from the values in the paper:

In [ ]:
M = np.array([4,-8,8])
K = np.array([-8,3,-6])

`pfv_to_flux(M, K)` converts the PFV flux integers $(M, K)$ into the full symplectic flux vector $[f \mid h] \in \mathbb{Z}^Q$ using the basis change and the coni-LCS structure. For $h^{1,2} = 3$, the result has length $Q = 2(h^{1,1}+1) = 200$ (though most components are zero for this simple example):

In [ ]:
flux = model.pfv_to_flux(M,K)
flux

We now derive the corresponding $F$-term minimum in two ways.

#### Starting from two-term racetrack approximation

Using $g_s = 0.38$ (i.e. $\tau_0 = i/g_s$) from [2009.03312], we call `pfv_to_moduli(M, K, τ)` to obtain the initial bulk moduli and conifold modulus from the PFV analytical solution. Notice that $z^1 = z_{\rm cf}$ is tiny ($\sim 5 \times 10^{-6}$) — the vacuum is deep inside the conifold regime:

In [ ]:
gs = 0.38
tau0 = 1j/gs
z0 = model.pfv_to_moduli(M,K,tau0)
z0

The PFV approximation satisfies $D_{z_{\rm cf}} W = 0$ by construction, but the remaining F-terms $D_{z^\alpha} W$ and $D_\tau W$ are only approximately zero (at the level of the racetrack truncation). The residuals confirm that this is a useful starting point but not yet the exact vacuum:

In [ ]:
model.DW(z0,jnp.conj(z0),tau0,jnp.conj(tau0),flux)

We use `scipy.optimize.root` with the **real representation** of the F-term system. The methods used are:
- `_convert_complex_to_real(z, z̄, τ, τ̄)` → `x ∈ ℝ^{2(h^{1,2}+1)}`: packs ${\rm Re}(z^a), {\rm Im}(z^a), {\rm Re}(\tau), {\rm Im}(\tau)$ into a flat array
- `DW_x(x, flux)` → `ℝ^{2(h^{1,2}+1)}`: evaluates $\{{\rm Re}(D_I W), {\rm Im}(D_I W)\}$ as a flat real vector
- `dDW_x(x, flux)`: the full Jacobian of `DW_x`, computed analytically via JAX autodiff (using `jac=dDW_x` avoids finite-difference approximation and significantly speeds up convergence)

In [ ]:
# Get array of real moduli values (for optimisation purposes)
x0 = model._convert_complex_to_real(z0,jnp.conj(z0),tau0,jnp.conj(tau0))

# Run scipy.optimize.root method to find solutions to DW=0
res = root(model.DW_x,x0,args=(flux,),jac=model.dDW_x,tol=1e-10,method="hybr")

res

After `scipy.optimize.root` converges (`res.success = True`), we recover the solution `x1 = res.x` in the real representation and verify that all F-terms are small. `DW_x` returns a real vector of length $2(h^{1,2}+1) = 8$ interleaving real and imaginary parts of $D_I W$:

In [ ]:
# Get solution
x1 = res.x

# Compute DW
DW = model.DW_x(x1,flux)

print("DW: ",np.abs(DW))

A physical solution must lie inside the Kähler cone. `model.periods.hyperplanes` stores the normal vectors $\mathbf{n}_I$ of the cone walls; the condition ${\rm Im}(z^a)\,(\mathbf{n}_I)_a > 0$ for all wall normals $I$ verifies that all relevant curve volumes are positive. We also check ${\rm Im}(\tau) > 0$ (physical string coupling $g_s = 1/{\rm Im}(\tau)$):

In [ ]:
z1,_,t1,_ = model._convert_real_to_complex(x1)

# Check Kähler cone: all hyperplane inequalities Im(z^a) n_a > 0 must be satisfied
flag = np.all(z1.imag@model.lcs_tree.hyperplanes.T>0)&(t1.imag>0)

print("Inside facet of Kähler cone: ",flag)
print(f"z_cf = {z1[0]:.3e}  (Im(z_cf) = {z1[0].imag:.3e})")
print(f"g_s  = {1.0/t1.imag:.4f}")

`superpotential(z, τ, flux, normalise=True)` divides by the period $X^0$ to give the gauge-invariant superpotential $W_0 = W / X^0$, which is independent of the overall period normalisation and directly comparable to literature values. Without `normalise=True`, the raw superpotential $W$ depends on the chosen normalisation of the symplectic basis.

In [ ]:
W1 = model.superpotential(z1,t1,flux,normalise=True)

print("W0: ",np.abs(W1))

This agree with the expected value $|W_0|\approx 6.9\times 10^{-4}$ stated in Eq. (4.59) of [2009.03312](https://arxiv.org/pdf/2009.03312).

#### Deriving PFV input

We now show how to derive the PFV starting point from scratch, without relying on values quoted in [2009.03312]. In the PFV approximation, `pfv_to_moduli(M, K, τ)` analytically satisfies $D_{z^a} W = 0$ for all complex structure moduli given $(M, K, \tau)$. The remaining condition is $D_\tau W = 0$, which is a two-dimensional real root-finding problem for ${\rm Re}(\tau)$ and ${\rm Im}(\tau)$.

We construct a JIT-compiled objective function that takes $(\tau_{\rm re}, \tau_{\rm im})$, evaluates the PFV moduli, and returns the last two components of `DW_x` (corresponding to ${\rm Re}(D_\tau W)$ and ${\rm Im}(D_\tau W)$):

In [ ]:
@jit
def obj(x,flux):
    
    tau0 = x[0]+1j*x[1]
    
    z0 = model.pfv_to_moduli(M,K,tau0)
    
    x0 = model._convert_complex_to_real(z0,jnp.conj(z0),tau0,jnp.conj(tau0))
    
    return model.DW_x(x0,flux)[-2:]


We find a minimum as follows

In [ ]:
x0 = np.array([0.,1.5])
            
pfv = root(obj,x0,args=(flux,),tol=1e-10,method="hybr")

pfv

This allows us to define the PFV solution as follows

In [ ]:
x1 = pfv.x
tau0 = x1[0]+1j*x1[1]
gs = 1/tau0.imag
z0 = model.pfv_to_moduli(M,K,tau0)

print("g_s at the PFV level: ",gs)

Note that this solution slightly deviates from the value $g_s=0.38$ stated in [2009.03312](https://arxiv.org/pdf/2009.03312) (also used above)
since we are not restricting to a simple 2-term racetrack.

At this initial guess, the $F$-term conditions are approximately satisfied

In [ ]:
model.DW(z0,jnp.conj(z0),tau0,jnp.conj(tau0),flux)

To find the true minimum, we can run the following lines to search for roots of $D_I W$:

In [ ]:
# Get array of real moduli values (for optimisation purposes)
x0 = model._convert_complex_to_real(z0,jnp.conj(z0),tau0,jnp.conj(tau0))

# Run scipy.optimize.root method to find solutions to DW=0
res = root(model.DW_x,x0,args=(flux,),jac=model.dDW_x,tol=1e-10,method="hybr")

res

Let us compute the value of the F-terms

In [ ]:
# Get solution
x1 = res.x

# Compute DW
DW = model.DW_x(x1,flux)

print("DW: ",np.abs(DW))

We check that the solution is inside the facet of the Kähler cone

In [ ]:
z1,_,t1,_ = model._convert_real_to_complex(x1)

flag = np.all(z1.imag@model.lcs_tree.hyperplanes.T>0)&(t1.imag>0)

print("Inside facet of Kähler cone: ",flag)

Compute value of $W_0$ with appropriate normalisation

In [ ]:
W1 = model.superpotential(z1,t1,flux,normalise=True)

print("W0: ",np.abs(W1))

This agree with the expected value $|W_0|\approx 6.9\times 10^{-4}$ stated in Eq. (4.59) of [2009.03312](https://arxiv.org/pdf/2009.03312).

## Bulk approximation at coniLCS

### Introduction

The above functions are implemented in the class `periods_coniLCS_bulk` which is automatically called in `periods` class if we specify `limit="coniLCS_bulk"`.

In many applications, it can be convenient to integrate out the conifold modulus and simply work with the bulk moduli $z^{\alpha}$.
To do so, we discard the logarithmic term in the prepotential
as well as all other higher order corrections arising from expanding around the conifold singularity.
We work with the effective bulk prepotential given by
\begin{equation}
    F_{\mathrm{eff. bulk}}(z) = F_{\mathrm{poly}}(z)+F_{\mathrm{inst}}(z)
\end{equation}
in terms of 
\begin{equation}
    F_{\mathrm{poly}}(z) = -\frac{1}{6}\widetilde{\kappa}_{abc}z^a z^b z^c+\frac{1}{2}a_{ab}z^a z^b
        +\tilde{b}_{a}z^a + \dfrac{\text{i}}{2}\tilde{\xi}\; , \quad 
    F_{\mathrm{inst}}(z) = -\frac{1}{(2\pi\mathrm{i})^3}\, \sum_{\mathbf{q}\in\mathcal{M}(\widetilde{X}), \mathbf{q}\neq \mathbf{q}_{\text{cf}}}\, 
    n_q^{0}\, \text{Li}_3\left (\text{e}^{2\pi \text{i} q_a z^a}\right )\, ,
\end{equation}
where
\begin{equation}
\tilde{b}_{a} = b_a + \dfrac{n_{\text{cf}}}{24} \delta_a^1\, .
\end{equation}

As we demonstrate below, this description is sufficient to derive the above minimum for the bulk complex structure moduli, while the conifold modulus can be effectively set to zero for most calculations.

### Example

We reproduce the same example from [2009.03312](https://arxiv.org/pdf/2009.03312), now using the bulk approximation (`limit="coniLCS_bulk"`). Compared to `limit="coniLCS"`, we also pass `use_gvs=True` to include worldsheet instanton corrections from GV invariants in the effective bulk prepotential — these are the non-conifold curves omitted in `limit="coniLCS"` near $z_{\rm cf}=0$. The conifold modulus is integrated out, and the model describes only the $h^{1,2}-1 = 2$ bulk moduli plus the axio-dilaton:

In [ ]:
from cytools import Polytope
poly = Polytope([[-1,3,-2,-1],[1,-1,0,0],[-1,0,0,1],[-1,0,0,0],[-1,0,1,1],[-1,0,2,0],[-1,0,1,0]])
cy = poly.triangulate().get_cy()

basis_matrix = np.array([[0, 1, 1], [1, 1, 0], [0, 0, 1]])
conifold_curve = np.array([-1,1,0])

bulkEFT = jvc.FluxVacuaFinder(h12=cy.h11(), Q=cy.h11()+cy.h12()+2, 
                                         use_cytools=True, 
                                         mirror_cy=cy, 
                                         ncf=2,
                                         use_gvs=True,
                                         maximum_degree=10,
                                         basis_change=basis_matrix,
                                         conifold_basis=True,
                                         conifold_curve = conifold_curve,
                                         prange=10,
                                         limit="coniLCS_bulk")

#bulkEFT

The $a$-matrix and $b$-vector should match those of the full coni-LCS model (the bulk approximation does not change the polynomial part of the prepotential, only removes the logarithmic term). Note that $\tilde{b}_1 = b_1 + n_{\rm cf}/24$, which adds $2/24 = 1/12$ to the first entry — this is the integrated-out correction from the conifold modulus:

In [ ]:
bulkEFT.lcs_tree.a_matrix,bulkEFT.lcs_tree.b_vector*24

The flux vector and PFV data are the same as in the full coni-LCS case. `pfv_to_flux` and `pfv_to_moduli` work identically in both limits since the PFV parametrisation only depends on the conifold structure:

In [ ]:
M = np.array([4,-8,8])
K = np.array([-8,3,-6])

flux = bulkEFT.pfv_to_flux(M,K)
flux

Define PFV data

In [ ]:
gs = 0.38
tau0 = 1j/gs
z0 = bulkEFT.pfv_to_moduli(M,K,tau0)
z0

In the bulk approximation the conifold modulus is fixed at $z_{\rm cf} = 0$. The real state vector `x` for the bulk degrees of freedom has length $2(h^{1,2}) = 6$ (real/imaginary parts of two bulk moduli plus the axio-dilaton). We prepend `jnp.zeros(2)` (representing ${\rm Re}(z_{\rm cf}) = {\rm Im}(z_{\rm cf}) = 0$) to call the full `DW_x`, then discard the first two components (which give $D_{z_{\rm cf}} W$, no longer relevant in the bulk approximation). The Jacobian `jac` is the corresponding $6\times 6$ sub-block:

In [ ]:
@jit
def obj(x,flux):
    
    return bulkEFT.DW_x(jnp.append(jnp.zeros(2),x),flux)[2:]

@jit
def jac(x,flux):
    
    return bulkEFT.dDW_x(jnp.append(jnp.zeros(2),x),flux)[2:,2:]

We can find a root of the F-term conditions as follows

In [ ]:
x0 = bulkEFT._convert_complex_to_real(z0,jnp.conj(z0),tau0,jnp.conj(tau0))[2:]
obj(x0,flux)

In [ ]:
res = root(obj,x0,args=(flux,),jac=jac,tol=1e-10,method="hybr")

res

Let us check explicitly the F-terms

In [ ]:
x1 = res.x

DW = obj(x1,flux)

print("DW: ",np.abs(DW))

For the bulk approximation, the Kähler cone check only applies to the bulk moduli ($z^\alpha$, $\alpha \geq 2$), since $z_{\rm cf}$ was set to zero. We project onto the bulk hyperplanes (those with nonzero components in the bulk directions) and check ${\rm Im}(\tau) > 0$:

In [ ]:
z1,_,t1,_ = bulkEFT._convert_real_to_complex(jnp.append(jnp.zeros(2),x1))

hypers = bulkEFT.lcs_tree.hyperplanes

flag = np.all(hypers[:,1:][np.any(hypers[:,1:]!=0,axis=1)]@z1.imag[1:]>0)&(t1.imag>0)

print("Inside facet of Kähler cone: ",flag)

The gauge-invariant superpotential is again computed with `normalise=True`. The bulk approximation should give essentially the same $|W_0|$ as the full coni-LCS model, since the conifold modulus contribution to $W$ is suppressed by $z_{\rm cf} \ln z_{\rm cf} \to 0$:

In [ ]:
W0 = bulkEFT.superpotential(z1,t1,flux,normalise=True)

print("W0: ",np.abs(W0))

We observe that this again matches the expectation of [2009.03312](https://arxiv.org/pdf/2009.03312).

## Series expansion of the conifold modulus

### Introduction

The `coniLCS_series` limit expands the full `coniLCS` prepotential in powers of $z_\text{cf}$ around $z_\text{cf} = 0$, up to some truncation order `nmax`. The key step is expanding the polylogarithm contributions associated with the conifold curve: since the conifold curve has charge $c = (1, 0, \ldots, 0)$ in the conifold basis, the corresponding instanton sum $\text{Li}_3(e^{2\pi i z_\text{cf}})$ develops a series in $z_\text{cf}$. This expansion is what generates the characteristic logarithmic behaviour

$$F_\text{coni}(z_\text{cf}) = \frac{{n}_\text{cf}}{2} \cdot \frac{z_\text{cf}^2}{2\pi \mathrm{i}} \log\!\left(-2\pi \mathrm{i}\, z_\text{cf}\right)$$

which is absent in the `coniLCS` and `coniLCS_bulk` limits. The remaining instanton contributions (from curves not proportional to the conifold curve) are regular at $z_\text{cf} = 0$ and are expanded straightforwardly.

The result is a prepotential of the form

$$F_\text{coni-LCS}(z_\text{cf}, z^a) = F_\text{coni}(z_\text{cf}) + \sum_{n=0}^{n_\text{max}} F^{(n)}_\text{LCS}(z^a)\, z_\text{cf}^n$$

where the coefficients $F^{(n)}_\text{LCS}(z^a)$ depend on the bulk moduli through polynomials and polylogarithms. The truncation order `nmax` (default 2) controls the accuracy: `nmax=2` gives linear accuracy at the level of the superpotential $W = \partial_i F$, which is sufficient for finding the exponentially small conifold VEV $|z_\text{cf}| \sim e^{-2\pi K / (n_\text{cf} g_s M)}$.

Unlike the bulk approximation, $z_\text{cf}$ is a dynamical variable here and is solved for alongside the bulk moduli and the axio-dilaton.


### Example

In [ ]:
from cytools import Polytope
poly = Polytope([[-1,3,-2,-1],[1,-1,0,0],[-1,0,0,1],[-1,0,0,0],[-1,0,1,1],[-1,0,2,0],[-1,0,1,0]])
cy = poly.triangulate().get_cy()

basis_matrix = np.array([[0, 1, 1], [1, 1, 0], [0, 0, 1]])
conifold_curve = np.array([-1,1,0])

linearisedEFT = jvc.FluxVacuaFinder(h12=cy.h11(), Q=cy.h11()+cy.h12()+2, 
                                         use_cytools=True, 
                                         mirror_cy=cy, 
                                         ncf=2,
                                         use_gvs=True,
                                         maximum_degree=10,
                                         basis_change=basis_matrix,
                                         conifold_basis=True,
                                         conifold_curve = conifold_curve,
                                         prange=10,
                                         limit="coniLCS_series")

The $a$-matrix and $b$-vector should match those of the full coni-LCS model (the series approximation does not change the polynomial part of the prepotential, only removes the logarithmic term). Note that $\tilde{b}_1 = b_1 + n_{\rm cf}/24$, which adds $2/24 = 1/12$ to the first entry — this is the integrated-out correction from the conifold modulus:

In [ ]:
linearisedEFT.lcs_tree.a_matrix,linearisedEFT.lcs_tree.b_vector*24

The flux vector and PFV data are the same as in the full coni-LCS case. `pfv_to_flux` and `pfv_to_moduli` work identically in both limits since the PFV parametrisation only depends on the conifold structure:

In [ ]:
M = np.array([4,-8,8])
K = np.array([-8,3,-6])

flux = linearisedEFT.pfv_to_flux(M,K)
flux

Define PFV data

In [ ]:
gs = 0.38
tau0 = 1j/gs
z0 = linearisedEFT.pfv_to_moduli(M,K,tau0)
z0

We can find a root of the F-term conditions as follows

In [ ]:
x0 = linearisedEFT._convert_complex_to_real(z0,jnp.conj(z0),tau0,jnp.conj(tau0))
linearisedEFT.DW_x(x0,flux)

In [ ]:
res = root(linearisedEFT.DW_x,x0,args=(flux,),jac=linearisedEFT.dDW_x,tol=1e-10,method="hybr")

res

Let us check explicitly the F-terms

In [ ]:
x1 = res.x

DW = linearisedEFT.DW_x(x1,flux)

print("DW: ",np.abs(DW))

Check that solution is inside the Kähler cone

In [ ]:
z1,_,t1,_ = linearisedEFT._convert_real_to_complex(x1)

hypers = linearisedEFT.lcs_tree.hyperplanes

flag = np.all(hypers@z1.imag>0)&(t1.imag>0)

print("Inside facet of Kähler cone: ",flag)

The gauge-invariant superpotential is again computed with `normalise=True`. The bulk approximation should give essentially the same $|W_0|$ as the full coni-LCS model, since the conifold modulus contribution to $W$ is suppressed by $z_{\rm cf} \ln z_{\rm cf} \to 0$:

In [ ]:
W0 = linearisedEFT.superpotential(z1,t1,flux,normalise=True)

print("W0: ",np.abs(W0))

We observe that this again matches the expectation of [2009.03312](https://arxiv.org/pdf/2009.03312).